In [ ]:
!pip install -q transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset
from collections import Counter

In [ ]:
dataset = load_dataset("ccdv/arxiv-classification")
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/3.30k [00:00<?, ?B/s]

data/train-00000-of-00004.parquet:   0%|          | 0.00/218M [00:00<?, ?B/s]

data/train-00001-of-00004.parquet:   0%|          | 0.00/215M [00:00<?, ?B/s]

data/train-00002-of-00004.parquet:   0%|          | 0.00/215M [00:00<?, ?B/s]

...

data/test-00000-of-00001.parquet:   0%|          | 0.00/74.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/28388 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 28388
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2500
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2500
    })
})

In [ ]:
def filter_cs_categories(dataset):
  allowed_labels = {
      1,  # cs.CV
      2,  # cs.AI
      3,  # cs.SY
      5,  # cs.CE
      6,  # cs.PL
      7,  # cs.IT
      8,  # cs.DS
      9   # cs.NE
  }

  for split in ["train", "validation", "test"]:
    dataset[split] = dataset[split].filter(lambda x: x["label"] in allowed_labels)

  return dataset

In [ ]:
dataset = filter_cs_categories(dataset)
dataset

In [ ]:
def remap_labels(dataset):
  remap_labels = {
      1: 0,
      2: 1,
      3: 2,
      5: 3,
      6: 4,
      7: 5,
      8: 6,
      9: 7
  }

  for split in ["train", "validation", "test"]:
    dataset[split] = dataset[split].map(lambda x: {"label": remap_labels[x["label"]]})

  return dataset

In [ ]:
dataset = remap_labels(dataset)
dataset

Map:   0%|          | 0/20752 [00:00<?, ? examples/s]

Map:   0%|          | 0/1819 [00:00<?, ? examples/s]

Map:   0%|          | 0/1842 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 20752
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1819
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1842
    })
})

In [ ]:
from collections import Counter

Counter(dataset["train"]["label"])

Counter({6: 3527,
         7: 2560,
         2: 2631,
         3: 2117,
         0: 2137,
         4: 2443,
         5: 2768,
         1: 2569})

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

In [ ]:
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, max_length=256)

In [ ]:
tokenized_dataset = dataset.map(tokenize_function, batched=True)

In [ ]:
print(tokenized_dataset["train"][0].keys())
print(len(tokenized_dataset["train"][0]["input_ids"]))

dict_keys(['text', 'label', 'input_ids', 'attention_mask'])
256


In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("answerdotai/ModernBERT-base", num_labels=8)

model

In [ ]:
print(model.config.num_labels)

8


In [ ]:
print(model.classifier)

Linear(in_features=768, out_features=8, bias=True)


In [ ]:
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

In [ ]:
print(tokenized_dataset["train"][0].keys())

dict_keys(['label', 'input_ids', 'attention_mask'])


In [ ]:
from transformers import TrainingArguments
training_args = TrainingArguments(
    output_dir="./modernbert_classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    logging_steps=100,
    report_to="none"
)

In [ ]:
import evaluate
accuracy = evaluate.load("accuracy")

In [ ]:
import numpy as np

def compute_metrics(eval_pred):

    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    return accuracy.compute(
        predictions=predictions,
        references=labels
    )

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.530582,0.515647,0.848268
2,0.414652,0.510896,0.851567


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy
1,0.530582,0.515647,0.848268
2,0.414652,0.510896,0.851567
3,0.278384,0.648520,0.850467


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=7782, training_loss=0.445406381650683, metrics={'train_runtime': 4994.5523, 'train_samples_per_second': 12.465, 'train_steps_per_second': 1.558, 'total_flos': 1.0607551445532672e+16, 'train_loss': 0.445406381650683, 'epoch': 3.0})

In [ ]:
results = trainer.evaluate()
print(results)

Training Loss,Validation Loss,Epoch,Accuracy
0.278384,0.510896,3,0.851567


{'eval_loss': 0.5108960866928101, 'eval_accuracy': 0.8515667949422759}


In [ ]:
test_results = trainer.evaluate(tokenized_dataset["test"])
print(test_results)

Training Loss,Validation Loss,Epoch,Accuracy
0.278384,0.448964,3,0.869164


{'eval_loss': 0.4489637613296509, 'eval_accuracy': 0.8691639522258415}


In [ ]:
trainer.save_model("modernbert_classifier")

In [ ]:
tokenizer.save_pretrained("modernbert_classifier")

In [ ]:
!zip -r modernbert_classifier.zip modernbert_classifier